**Navigation** : [Index](README.md) | [<< RL-2 Wrappers](rl_2_wrappers_sauvegarde_callbacks.ipynb) | [RL-4 Bandits >>](rl_4_multi_armed_bandits.ipynb)
# Notebook 3 – Hindsight Expérience Replay (HER) et Sauvegarde Avancée

**Serie** : Reinforcement Learning | **Notebook** : 3/13 | **Duree estimee** : 40-45 min

Dans ce troisième notebook, nous allons aborder plusieurs sujets avancés :

1. **Hindsight Expérience Replay (HER)**, une technique permettant d'entraîner un agent sur des tâches de type "Goal-Conditioned RL" (où l’agent doit atteindre un objectif paramétrable),
2. un **exemple pratique** avec l’environnement `parking-v0` (issu de la bibliothèque [highway-env](https://github.com/eleurent/highway-env)),
3. la **sauvegarde/chargement avancés** d’un modèle, incluant la sauvegarde et le rechargement de la buffer d’expérience (replay buffer).

Cet environnement `parking-v0` est un cas classique d’apprentissage par renforcement à but : la récompense dépend d’un objectif (ici, se garer à un endroit précis, avec la bonne orientation).

Nous verrons comment utiliser HER avec différents algorithmes (SAC, DDPG), comment **sauvegarder/recharger** un modèle **et** sa mémoire de rejouage, et comment **évaluer** l’agent.


**Rappel sur le “Goal-Conditioned RL”**  
- Dans certaines tâches, un “goal” (objectif) fait partie de l’état désiré. Par exemple, la position finale d’un bras robotique, la place de parking à occuper, etc.  
- Les observations se présentent souvent comme un dictionnaire : `{'observation': ..., 'desired_goal': ..., 'achieved_goal': ...}`.  
- HER (Hindsight Expérience Replay) ré-étiquette des transitions a posteriori pour rendre l’apprentissage plus efficace, surtout quand la récompense est très clairsemée (sparse).


## Installation des dépendances

Sous Windows, nous n’utilisons pas de commandes `apt-get`. Nous procédons uniquement par `pip` pour installer :

- Stable-Baselines3 (avec le `[extra]`),
- highway-env pour l’environnement `parking-v0`,
- (Optionnel) `moviepy` pour l’enregistrement de vidéos.

```bash
pip install "stable-baselines3[extra]>=2.0.0a4"
pip install highway-env
pip install moviepy
```

Dans un notebook Python, on peut faire :
```python
%pip install "stable-baselines3[extra]>=2.0.0a4" highway-env moviepy
```

In [1]:
# Installation par commande magique Notebook (Windows-friendly, pas de apt-get)


## Imports Essentiels

Nous importons :
- `HerReplayBuffer` : le buffer de rejouage spécialisé pour HER,
- des algorithmes (SAC, DDPG) compatibles avec HER (il faut un algo off-policy pour combiner avec HER),
- l’environnement `parking-v0` depuis highway_env,
- NumPy, etc.


> *Ancres savantes -- Andrychowicz, M., Wolski, F., Ray, A., Schneider, J., Fong, R., Welinder, P., McGrew, B., Tobin, J., Abbeel, P. & Zaremba, W. (2017), Hindsight Expérience Replay, NeurIPS 2017, arXiv:1707.01495 (HER, re-etiquetage des echecs en succes synthetiques pour apprentissage goal-conditioned multi-objectif) ; Haarnoja, T., Zhou, A., Abbeel, P. & Levine, S. (2018), Soft Actor-Critic, ICML 2018, arXiv:1801.01290 (SAC, acteur-critique off-policy a entropie maximale) ; Lillicrap, T.P., Hunt, J.J., Pritzel, A., Heess, N., Erez, T., Tassa, Y., Silver, D. & Wierstra, D. (2016), Continuous Control with Deep Reinforcement Learning, ICLR 2016, arXiv:1509.02971 (DDPG, acteur-critique déterministe off-policy pour actions continues).*

In [2]:
import gymnasium as gym
import highway_env
import numpy as np

from stable_baselines3 import HerReplayBuffer, SAC, DDPG
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.evaluation import evaluate_policy

print("Imports OK")

Imports OK


### Environnement Parking

[`parking-v0`](https://github.com/eleurent/highway-env#parking-env) est un environnement « goal-conditioned » : la position et l’orientation cibles font partie de l’`info['goal']`. Pour résoudre cette tâche, on doit apprendre à manœuvrer la voiture pour qu’elle se gare.

![parking-env](https://raw.githubusercontent.com/eleurent/highway-env/gh-media/docs/media/parking-env.gif)


### Création de l’environnement Gym

In [3]:
env = gym.make("parking-v0")
obs, _ = env.reset()
print("Observation :", obs.keys())
print("Exemple d'observation['observation']:", obs["observation"].shape)
print("Exemple d'observation['desired_goal']:", obs["desired_goal"].shape)

Observation : odict_keys(['observation', 'achieved_goal', 'desired_goal'])
Exemple d'observation['observation']: (6,)
Exemple d'observation['desired_goal']: (6,)


Par défaut, l’action est continue (2 dimensions : accélération et direction). On peut vérifier en imprimant `env.action_space` ou `env.observation_space`.

**Structure de l’observation**  
- `obs['observation']`: informations sur la voiture (position, vitesse, angle...).  
- `obs['desired_goal']`: position/angle cible (le “parking spot”).  
- `obs['achieved_goal']`: l’état effectivement atteint par la voiture.  

La récompense dépend souvent de la distance entre `achieved_goal` et `desired_goal`. HER va ré-étiqueter certains buts pour générer des transitions artificiellement “réussies”.


## Entraîner un agent SAC avec HER

La configuration de `HerReplayBuffer` est centrale ici. Nous choisissons :
- `goal_selection_strategy="future"` (la stratégie la plus courante, on va remplacer le but original par un but futur observé dans le même épisode),
- `n_sampled_goal=4` (on crée 4 transitions artificielles par transition réelle),
- des hyperparamètres un peu custom pour *SAC* : `batch_size`, `policy_kwargs`, etc.

Au final, l’entraînement dure un certain temps (on peut ajuster le `total_timesteps` en fonction de la machine).




In [4]:
model_sac = SAC(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
    ),
    # on attend 1000 pas avant d'entraîner,
    # afin d'avoir au moins un épisode complet stocké.
    learning_starts=1000,  
    buffer_size=50000,
    batch_size=64,
    policy_kwargs=dict(net_arch=[64, 64]),
    train_freq=1,
    gradient_steps=1,
    verbose=1,
)
model_sac.learn(total_timesteps=5000, log_interval=100)



# Sauvegarde du modèle ET de la replay buffer avant suppression
model_sac.save("her_sac_parking")
model_sac.save_replay_buffer("her_sac_parking_replay_buffer")
del model_sac  # On supprime de la RAM



Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.



**Focus sur la stratégie `goal_selection_strategy=\"future\"`**  
- “future” signifie qu’on va remplacer le but initial par un but échantillonné **plus tard** dans la même trajectoire.  
- Cela favorise l’apprentissage, car beaucoup d’états futurs atteints sont convertis en “objectifs cibles”.  
- Alternatives : “final”, “episode”, “random” — à tester selon l’environnement.


## Rechargement du modèle et évaluation

Nous rechargeons ensuite le modèle, et on peut l’évaluer sur quelques épisodes :


In [5]:
from stable_baselines3.common.monitor import Monitor

# Rechargement
model_sac = SAC.load("her_sac_parking", env=env)

# Évaluation

eval_env = Monitor(env)  # Ajout du Monitor pour éviter les warnings
mean_reward, std_reward = evaluate_policy(model_sac, eval_env, n_eval_episodes=10, deterministic=True)

print(f"SAC Parking : reward moyen={mean_reward:.2f} +/- {std_reward:.2f}")



Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


SAC Parking : reward moyen=-56.64 +/- 22.18


La notion de « récompense » dans un environnement `goal-conditioned` (HER) reflète la distance à l’objectif et la réussite/échec à se garer. On peut inspecter `info.get("is_success", False)` pour savoir si l’épisode est terminé avec succès.

## Exemple avec DDPG

Nous pouvons reproduire la même idée avec un autre algorithme off-policy (DDPG). On ajoute souvent un bruit d’exploration, `NormalActionNoise` :

In [6]:
# On crée un bruit gaussien pour l’action
n_actions = env.action_space.shape[0]  # en général = 2
noise_std = 0.2
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=noise_std * np.ones(n_actions))

model_ddpg = DDPG(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
    ),
    verbose=1,
    # On réduit la taille de la buffer
    buffer_size=50_000,
    learning_rate=1e-3,
    action_noise=action_noise,
    gamma=0.95,
    # batch_size plus petit
    batch_size=64,
    # Réseau plus léger
    policy_kwargs=dict(net_arch=[64, 64]),
    # On attend un peu avant d'entraîner
    learning_starts=1000,
    # On fait 1 step d'entraînement par step environnement
    train_freq=1,
    gradient_steps=1,
)

# On ne va pas jusqu'à 2e5 steps
# mais 5000 ou 10 000 pour une démo rapide
model_ddpg.learn(10_000)  # par exemple

# Sauvegarde
model_ddpg.save("her_ddpg_parking")
del model_ddpg


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 106      |
|    ep_rew_mean     | -48      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 59       |
|    time_elapsed    | 7        |
|    total_timesteps | 425      |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 88.8     |
|    ep_rew_mean     | -42.8    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 8        |
|    fps             | 61       |
|    time_elapsed    | 11       |
|    total_timesteps | 710      |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 89.1     |
|    ep_rew_mean     | -46      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 12       |
|    fps             | 57       |
|    time_elapsed    | 18       |
|    total_timesteps | 1069     |
| train/             |          |
|    actor_loss      | 0.277    |
|    critic_loss     | 0.435    |
|    learning_rate   | 0.001    |
|    n_updates       | 68       |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 79.1     |
|    ep_rew_mean     | -41      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 16       |
|    fps             | 47       |
|    time_elapsed    | 26       |
|    total_timesteps | 1265     |
| train/             |          |
|    actor_loss      | 0.463    |
|    critic_loss     | 0.0135   |
|    learning_rate   | 0.001    |
|    n_updates       | 264      |
---------------------------------


...

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 79       |
|    ep_rew_mean     | -39.9    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 108      |
|    fps             | 25       |
|    time_elapsed    | 332      |
|    total_timesteps | 8613     |
| train/             |          |
|    actor_loss      | 2.35     |
|    critic_loss     | 0.0192   |
|    learning_rate   | 0.001    |
|    n_updates       | 7612     |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 81       |
|    ep_rew_mean     | -40.4    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 112      |
|    fps             | 25       |
|    time_elapsed    | 355      |
|    total_timesteps | 9170     |
| train/             |          |
|    actor_loss      | 2.49     |
|    critic_loss     | 0.0685   |
|    learning_rate   | 0.001    |
|    n_updates       | 8169     |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 80.2     |
|    ep_rew_mean     | -40.1    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 116      |
|    fps             | 25       |
|    time_elapsed    | 360      |
|    total_timesteps | 9283     |
| train/             |          |
|    actor_loss      | 2.46     |
|    critic_loss     | 0.0171   |
|    learning_rate   | 0.001    |
|    n_updates       | 8282     |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 78       |
|    ep_rew_mean     | -38.9    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 120      |
|    fps             | 25       |
|    time_elapsed    | 367      |
|    total_timesteps | 9458     |
| train/             |          |
|    actor_loss      | 2.19     |
|    critic_loss     | 0.0149   |
|    learning_rate   | 0.001    |
|    n_updates       | 8457     |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 75.9     |
|    ep_rew_mean     | -37.5    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 124      |
|    fps             | 25       |
|    time_elapsed    | 372      |
|    total_timesteps | 9578     |
| train/             |          |
|    actor_loss      | 2.26     |
|    critic_loss     | 0.0249   |
|    learning_rate   | 0.001    |
|    n_updates       | 8577     |
---------------------------------


Chargement du modèle DDPG entraine avec HER pour comparaison.

In [7]:
model_ddpg = DDPG.load("her_ddpg_parking", env=env)

eval_env = Monitor(env)  # Ajout du Monitor pour eviter les warnings
mean_reward, std_reward = evaluate_policy(model_ddpg, eval_env, n_eval_episodes=10, deterministic=True)
print(f"DDPG Parking : reward moyen={mean_reward:.2f} +/- {std_reward:.2f}")


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


DDPG Parking : reward moyen=-15.83 +/- 2.29


## Sauvegarde et Chargement de la Replay Buffer

Une fonctionnalité avancée de Stable-Baselines3 est la **possibilité de sauvegarder aussi la buffer de rejouage** (replay buffer) :

- Par défaut, `model.save(...)` **ne** sauvegarde **pas** la replay buffer, car celle-ci peut être très volumineuse (plusieurs Go si on utilise des images, par ex.).
- Mais on peut la sauvegarder à part avec `model.save_replay_buffer(path)`, puis la recharger avec `model.load_replay_buffer(path)`.

Cela permet de **reprendre un entraînement** où on l’avait laissé, en conservant tout l’historique d’expériences collectées. Sur des environnements complexes, c’est très utile pour éviter de tout recommencer !

Exemple :

In [8]:
# Rechargement du modele + de sa replay buffer (sauvegardee a l'entrainement)

model_sac_2 = SAC.load("her_sac_parking", env=env)
print("Taille de la replay buffer AVANT rechargement :", model_sac_2.replay_buffer.size())

# La derniere trajectoire du pkl etait incomplete a la sauvegarde ;
# on la conserve entiere (pas de truncation) : pas de UserWarning SB3.
model_sac_2.load_replay_buffer("her_sac_parking_replay_buffer", truncate_last_traj=False)
print("Taille de la replay buffer APRES rechargement :", model_sac_2.replay_buffer.size())


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Taille de la replay buffer AVANT rechargement : 0
Taille de la replay buffer APRES rechargement : 5000



**Reprendre l’entraînement**  
- Après avoir chargé le modèle et la replay buffer, on peut appeler `model_sac_2.learn(N)` pour continuer exactement là où l’on s’était arrêté.  
- Utile pour *checkpoint* l’entraînement de temps en temps, sans perdre l’historique des transitions passées (particulièrement en off-policy).


## Enregistrement vidéo sous Windows

Comme évoqué dans les notebooks précédents, sous Windows, pas besoin de démarrer un display virtuel via `xvfb`. On peut simplement enregistrer en mode `rgb_array`. 

Voici une fonction utilitaire pour enregistrer une vidéo d’un agent (la même que dans les notebooks précédents, adaptée pour Windows) :

In [9]:
import base64
from pathlib import Path
from IPython.display import HTML

from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv

def record_video(env_id, model, video_length=1000, prefix="", video_folder="videos/"):
    # On crée un DummyVecEnv pour enregistrer
    eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
    
    # On active le recorder vidéo :
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset()
    for _ in range(video_length):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, info = eval_env.step(action)

    eval_env.close()

def show_videos(video_path="videos", prefix=""):
    """
    Affiche toutes les vidéos mp4 dans le dossier spécifié.
    """
    mp4_list = list(Path(video_path).glob(f"{prefix}*.mp4"))
    if len(mp4_list) == 0:
        print("Aucune vidéo trouvée.")
        return

    html_video = ""
    for mp4 in mp4_list:
        video_b64 = base64.b64encode(mp4.read_bytes()).decode("ascii")
        html_video += f"<video alt='{mp4}' autoplay loop controls style='height: 400px;'>\n" \
                      f"<source src='data:video/mp4;base64,{video_b64}' type='video/mp4' />\n" \
                      "</video>\n"
    return HTML(html_video)

print("Fonctions video enregistrees (record_video, show_videos)")

Fonctions video enregistrees (record_video, show_videos)


Pour tester, nous pouvons enregistrer une vidéo de `model_sac` ou `model_ddpg`. (Attention : `parking-v0` peut boucler un peu longtemps si on met un `video_length` trop grand.)

In [10]:
import contextlib
import io

# Le logger de VecVideoRecorder imprime un chemin absolu machine ;
# on capture ce stdout et on affiche un chemin relatif reproductible.
with contextlib.redirect_stdout(io.StringIO()):
    record_video("parking-v0", model_sac, video_length=500, prefix="sac-parking")

video_files = [p.name for p in Path("videos").glob("sac-parking*.mp4")]
print("Video enregistree : " + ", ".join(video_files))

show_videos("videos", prefix="sac-parking")


frame_index:   0%|          | 0/501 [00:00<?, ?it/s, now=None]

frame_index:  15%|█▍        | 75/501 [00:00<00:00, 663.53it/s, now=None]

frame_index:  34%|███▍      | 172/501 [00:00<00:00, 829.78it/s, now=None]

frame_index:  54%|█████▎    | 269/501 [00:00<00:00, 890.59it/s, now=None]

frame_index:  79%|███████▉  | 397/501 [00:00<00:00, 1035.55it/s, now=None]

Video enregistree : sac-parking-step-0-to-step-500.mp4


Vous devriez voir la voiture tenter de se garer…

## Exercices

Les exercices suivants approfondissent les concepts de HER et d'apprentissage par renforcement objectif-conditionne (goal-conditioned RL) abordes dans ce notebook.

### Exercice 1 : Comparaison des stratégies de sélection de buts

HER propose plusieurs stratégies pour re-etiqueter les transitions : `"future"`, `"final"` et `"episode"`. L'objectif est de comparer ces trois stratégies sur l'environnement `parking-v0` et d'observer leurs effets sur la vitesse de convergence et la performance finale.

**Indice** : Modifiez le paramètre `goal_selection_strategy` dans `replay_buffer_kwargs`. Entrainez chaque configuration avec le même nombre de `total_timesteps` (par exemple 5000) et comparez les recompenses moyennes.

In [11]:
# Exercice 1 : Comparaison des strategies de selection de buts
# TODO etudiant : Comparez "future", "final" et "episode" sur parking-v0 avec HER+SAC
# Indice : strategies = ["future", "final", "episode"]
# Indice : Pour chaque strategie, modifier goal_selection_strategy dans replay_buffer_kwargs
# Etape 1 : Definir les 3 strategies a tester
# Etape 2 : Boucler, creer SAC + HerReplayBuffer pour chaque strategie, entrainer (5000 steps)
# Etape 3 : Evaluer et stocker les recompenses, puis afficher un tableau comparatif

strategy_results = {}  # TODO etudiant : remplir avec {strategy: mean_reward}
print("Exercice a completer : comparaison des strategies de selection de buts")

Exercice a completer : comparaison des strategies de selection de buts


### Exercice 2 : Comparaison SAC vs DDPG avec HER

SAC et DDPG sont deux algorithmes off-policy compatibles avec HER, mais ils différent dans leur approche (SAC maximise l'entropie pour encourager l'exploration, DDPG utilise un bruit d'action explicite). Comparez leurs performances avec HER sur `parking-v0`.

**Indice** : Reprenez les configurations SAC et DDPG des sections précédentes. Entrainez les deux avec le même `total_timesteps` et comparez les courbes de recompense avec `matplotlib`.

In [12]:
# Exercice 2 : Comparaison SAC vs DDPG avec HER
# TODO etudiant : Entrainez SAC et DDPG (avec HER) sur parking-v0, comparez les courbes
# Indice : Reprenez les modeles des sections precedentes (HerReplayBuffer, meme timesteps)
# Indice : Stockez les rewards d'evaluation a intervalles reguliers pour tracer les courbes
# Etape 1 : Creer et entrainer SAC + HER (meme config que la section correspondante)
# Etape 2 : Creer et entrainer DDPG + HER (meme config que la section correspondante)
# Etape 3 : Evaluer les deux modeles et tracer les courbes de recompense comparatives

algo_results = {}  # TODO etudiant : remplir avec {"SAC": mean_reward, "DDPG": mean_reward}
print("Exercice a completer : comparaison SAC vs DDPG avec HER")

Exercice a completer : comparaison SAC vs DDPG avec HER


### Exercice 3 : Modification de la fonction de recompense

La fonction de recompense par defaut de `parking-v0` depend de la distance a l'objectif. L'objectif est de créer un wrapper qui penalise plus agressivement la distance a l'objectif, en utilisant par exemple une penalite proportionnelle au carre de la distance. Observez comment cette modification affecte le comportement de l'agent.

**Indice** : Créez un `gym.Wrapper` qui intercepte l'appel `step()` et recalcule la recompense en fonction de la distance entre `achieved_goal` et `desired_goal`. Utilisez `np.linalg.norm()` pour calculer la distance et renvoyez une penalite proportionnelle a `-distance**2`.

In [13]:
# Exercice 3 : Modification de la fonction de recompense
# TODO etudiant : Creez un wrapper qui penalise plus agressivement la distance a l'objectif
# Indice : class AggressiveRewardWrapper(gym.Wrapper):
# Indice : Dans step(), recalculez reward = -np.linalg.norm(achieved - desired)**2
# Etape 1 : Definir le wrapper avec __init__ et step
# Etape 2 : Intercepter l'observation dans step() pour recalculer la recompense
# Etape 3 : Entrainer SAC+HER avec le wrapper et comparer avec le reward par defaut

# class AggressiveRewardWrapper(gym.Wrapper):
#     def __init__(self, env, penalty_scale=1.0):
#         ...
#     def step(self, action):
#         obs, reward, terminated, truncated, info = self.env.step(action)
#         distance = np.linalg.norm(obs["achieved_goal"] - obs["desired_goal"])
#         reward = ...  # TODO etudiant : definir la nouvelle recompense
#         return obs, reward, terminated, truncated, info

print("Exercice a completer : modification de la fonction de recompense")

Exercice a completer : modification de la fonction de recompense


## Conclusion

Dans ce troisième notebook, nous avons abordé des fonctionnalités avancées de Stable-Baselines3 :

- **Hindsight Expérience Replay (HER)**, qui permet d’apprendre efficacement sur des tâches à but (objectif) en ré-étiquetant des transitions passées,
- l’utilisation de **SAC** ou **DDPG** avec HER (algorithmes off-policy),
- la **sauvegarde et le rechargement** de la replay buffer pour reprendre un entraînement ultérieurement,
- l’enregistrement vidéo « friendly pour Windows », sans dépendances `apt-get`.

Avec cela, vous disposez d’une base solide pour traiter des tâches plus complexes en Apprentissage par Renforcement, où l’agent doit atteindre des objectifs spécifiques.

---
**Retour au sommaire** : [Index RL](README.md)


### References academiques

- Andrychowicz, M., Wolski, F., Ray, A., Schneider, J., Fong, R., Welinder, P., McGrew, B., Tobin, J., Abbeel, P. & Zaremba, W. (2017). Hindsight Expérience Replay. NeurIPS 2017. arXiv:1707.01495.
- Haarnoja, T., Zhou, A., Abbeel, P. & Levine, S. (2018). Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor. ICML 2018. arXiv:1801.01290.
- Lillicrap, T.P., Hunt, J.J., Pritzel, A., Heess, N., Erez, T., Tassa, Y., Silver, D. & Wierstra, D. (2016). Continuous Control with Deep Reinforcement Learning. ICLR 2016. arXiv:1509.02971.
- Sutton, R.S. & Barto, A.G. (2018). Reinforcement Learning: An Introduction (2nd ed.). MIT Press.

